In [ ]:
etherpad_url = "http://localhost:9001/"
migration_pad_title = "Solr Migration Test Pad"
default_result_path = None
close_on_fail = False
transition_timeout = 10000

# Solr Volume Migration Test - Phase 1: Create a pad on the old Solr

Creates a pad while the stack runs against the old Solr and verifies it is searchable.

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

In [ ]:
import importlib

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)

## Open etherpad_url and check if the search box is editable.

In [ ]:
index_page = None

async def _step(page):
    await page.goto(etherpad_url)

    await expect(page.locator(".hashview-search-box")).to_be_editable()

    global index_page
    index_page = page

await run_pw(_step)

## Create a pad with migration_pad_title.

In [ ]:
import re

async def _step(page):
    await page.locator(".hashview-search-box").fill(migration_pad_title)
    await expect(page.locator('//button[text()="Create"]')).to_be_enabled()

    popup_future = page.wait_for_event('popup')
    await page.locator('//button[text()="Create"]').click()
    popup = await popup_future
    await expect(popup).to_have_title(re.compile(f"^{migration_pad_title}"), timeout=transition_timeout)

await run_pw(_step)

## Check that the pad is searchable.

In [ ]:
import asyncio

async def _step(page):
    for attempt in range(10):
        await index_page.reload()
        if await index_page.locator(f'text="{migration_pad_title}"').count() > 0:
            break
        await asyncio.sleep(3)
    await expect(index_page.locator(f'text="{migration_pad_title}"')).to_be_visible()

    return index_page

await run_pw(_step)

Clean up

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}